In [1]:
import json
import os
from pathlib import Path
from googleapiclient.discovery import build
from dotenv import load_dotenv

c:\Users\hp\OneDrive\Documents\Kuliah CS\SKRIPSI\judol\github\env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.4) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# KONFIGURASI
load_dotenv() # load_dotenv(dotenv_path="path/ke/.env")

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY belum ditemukan di file .env")


# url video: https://www.youtube.com/watch?v=zqv7ENToios
# VIDEO_ID = "zqv7ENToios"
VIDEO_ID = "zqv7ENToios"

# Inisialisasi YouTube API
youtube = build("youtube", "v3", developerKey=API_KEY)

In [3]:
# FUNCTION SCRAPE COMMENTS

def get_youtube_comments(video_id):
    
    comments_data = []

    request = youtube.commentThreads().list(
        part="snippet,replies",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request:

        response = request.execute()

        for item in response["items"]:

            comment = item["snippet"]["topLevelComment"]["snippet"]

            comment_info = {
                "comment_id": item["snippet"]["topLevelComment"]["id"],
                "author": comment["authorDisplayName"],
                "comment_text": comment["textDisplay"],
                "like_count": comment["likeCount"],
                "published_at": comment["publishedAt"],
                "reply_count": item["snippet"]["totalReplyCount"],
                "replies": []
            }

            # Ambil replies jika ada
            if "replies" in item:
                for reply in item["replies"]["comments"]:

                    reply_snippet = reply["snippet"]

                    reply_info = {
                        "reply_id": reply["id"],
                        "author": reply_snippet["authorDisplayName"],
                        "reply_text": reply_snippet["textDisplay"],
                        "like_count": reply_snippet["likeCount"],
                        "published_at": reply_snippet["publishedAt"]
                    }

                    comment_info["replies"].append(reply_info)

            comments_data.append(comment_info)

        request = youtube.commentThreads().list_next(
            request,
            response
        )

    return comments_data

In [4]:
# JALANKAN SCRAPER

print("Mengambil komentar...")

comments = get_youtube_comments(VIDEO_ID)

print(f"Total komentar diambil: {len(comments)}")

Mengambil komentar...
Total komentar diambil: 331


In [ ]:
# SIMPAN KE JSON

output_file = f"output/{VIDEO_ID}.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(comments, f, indent=4, ensure_ascii=False)

print(f"Data berhasil disimpan ke {output_file}")

Data berhasil disimpan ke zqv7ENToios.json
